In [16]:
import pdfplumber
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', None)

In [17]:
with pdfplumber.open("../data/raw/Ts/account-report-MKI-February-2026.pdf") as pdf:
    print(f"Total Pages in PDF: {len(pdf.pages)}")
    
    page1 = pdf.pages[0]
    
    lines = page1.extract_text_lines()
    fulltext = page1.extract_table()
    state_name = "unknown"
    month_name = "unknown"
    for line in lines:
        text = line["text"]
        
        # 1. Extract State Name
        if "Government of" in text:
            state_parts = text.split("of")
            if len(state_parts) > 1:
                state_name = state_parts[1].split("State")[0].strip()
        
        # 2. Extract Month and year
        # Looking for the line: "Monthly Key Indicators for the month of February 2026"
        if "month of" in text:
            month_parts = text.split("month of")
            if len(month_parts) > 1:
                month_name = month_parts[1]
        
        # 3. Stop if both are found
        if state_name != "unknown" and month_name != "unknown":
            break

    # Extracting Table Data
    table_data = page1.extract_table()
    
    print(f"State: {state_name}")
    print(f"Month: {month_name}")

Total Pages in PDF: 9
State: Telangana
Month:  February 2026


In [18]:
clean_headers =[
    "Description",
    "Budget Estimates 2025-26",
    "Actuals up to Feb 2026",
    "Percentage: Current 2025-26",
    "Percentage: Corresponding 2024-25"
]

df = pd.DataFrame(fulltext[2:],columns=clean_headers)


df["State"] =state_name
df["month"]=month_name
#df["year"]=year_pdf


df.head()

,Description,Budget Estimates 2025-26,Actuals up to Feb 2026,Percentage: Current 2025-26,Percentage: Corresponding 2024-25,State,month
0,1 REVENUE RECEIPTS,229720.63,154346.85,67.19,61.53,Telangana,February 2026
1,a)Tax Revenue,175319.36,139100.81,79.34,75.46,Telangana,February 2026
2,i)Goods and Service Tax,59704.59,48036.72,80.46,79.26,Telangana,February 2026
3,ii)Stamps and Regstration,19087.26,13775.51,72.17,38.58,Telangana,February 2026
4,iii)Land Revenue,11.20,0.54,4.84,7.39,Telangana,February 2026


In [19]:
remove_row = [
    "1 REVENUE RECEIPTS",
        "a)Tax Revenue",
    "2 CAPITAL RECEIPTS",
    "3 TOTAL RECEIPTS",
    "4 Revenue Expenditure (a+b+c+d+e)",
    "5 Capital Expenditure (a+b+c)",
    "7 TOTAL EXPENDITURE [4+5] (***)",
    "7 TOTAL EXPENDITURE [4+5] (****)",
    "a)Revenue",
    "b)Capital",
    "6 Sector wise Expediture (i+ii+iii+iv) (***)",
    "6 Sector wise Expediture (i+ii+iii+iv) (****)"
    
]
df = df[~df["Description"].isin(remove_row)]

tax_rev_list = [
    "i)Goods and Service Tax",
    "ii)Stamps and Regstration",
    "iii)Land Revenue",
    "iv)Sales Tax",
    "v)State Excise Duties",
    "vi)States Share of Union Taxes (*)",
    "vii)Other Taxes and duties ($)",
    "vii)Other Taxes and duties"
    ]

# Use .loc[rows, column] = value
df.loc[df["Description"].isin(tax_rev_list), "cat2"] = "Tax Revenue"

df.loc[~df["Description"].isin(tax_rev_list), "cat2"] = df["Description"].str[2:]

#1 REVENUE RECEIPTS
rev_rec_list =[
    "i)Goods and Service Tax",
    "ii)Stamps and Regstration",
    "iii)Land Revenue",
    "iv)Sales Tax",
    "v)State Excise Duties",
    "vi)States Share of Union Taxes (*)",
    "vii)Other Taxes and duties ($)",
    "b)Non-Tax Revenue",
    "c)Grant in aid and Contributions",
    "vii)Other Taxes and duties"
]

df.loc[df["Description"].isin(rev_rec_list),"cat1"] = "REVENUE RECEIPTS"


Cap_rec_list = [
    "a)Recovery of Loans and Advances",
    "b)Other Receipts",
    "c)Borrowings & Other Liabilities (Net ) (**)",
    "c)Borrowings & Other Liabilities (Net ) (**) (***)"
]

df.loc[df["Description"].isin(Cap_rec_list),"cat1"]="Capital Receipts"

df.loc[df["Description"].isin(Cap_rec_list),"cat2"] = df["Description"].str[2:]


rec_exp_list =[
    "(a)Expenditure on Revenue Account (excluding b,c,d,e)",
    "(b)Expenditure in Interest Payments",
    "(c)Expenditure on Salaries/Wages",
    "(d)Expenditure on Pension",
    "(e)Expenditure on Subsidy"
]

df.loc[df["Description"].isin(rec_exp_list),"cat1"]="Revenue Expenditure" 

df.loc[df["Description"].isin(rec_exp_list),"cat2"] = df["Description"].str[3:]

cap_exp_list =[
    "(a)Expenditute on Capital Account (excluding b)",
    "(b)Expenditure on Salaries/Wages",
    "(c)Inter State Settlement (Net)"
]

df.loc[df["Description"].isin(cap_exp_list),"cat1"] ="Capital Expenditure"
df.loc[df["Description"].isin(cap_exp_list),"cat2"] = df["Description"].str[3:]
#


#6 Sector wise Expediture (i+ii+iii+iv) (***)
#rev_cap=[   "a)Revenue","b)Capital"]

sec_exp_list =[
    "(i)General Sector",
    "(ii)Social Sector",
    "(iii)Economic Sector",
    "(iv)Grant in aid and Contributions"
]

df.loc[df["Description"].isin(sec_exp_list),"cat1"] ="Sector wise Expediture"
df.loc[df["Description"] == "(i)General Sector","cat2"] = df["Description"].str[3:]
df.loc[df["Description"] == "(ii)Social Sector","cat2"] = df["Description"].str[4:]
df.loc[df["Description"] == "(iii)Economic Sector","cat2"] = df["Description"].str[5:]
df.loc[df["Description"] == "(iv)Grant in aid and Contributions","cat2"] = df["Description"].str[4:]


#8 LOANS AND ADVANCES DISBURSED#
df.loc[df["Description"] == "8 LOANS AND ADVANCES DISBURSED","cat1"] = "LOANS AND ADVANCES DISBURSED"

#Revenue Surplus/Deficit [1-4]
df.loc[df["Description"] == "9 Revenue Surplus/Deficit [1-4]","cat1"] = "Revenue Surplus/Deficit(Rec-Exp)"

#10 Fiscal Deficit [1+2(a)+(b)] -(7+8)
df.loc[df["Description"] == "10 Fiscal Deficit [1+2(a)+(b)] -(7+8)","cat1"] = "Fiscal Deficit"

df.loc[df["Description"] == "10 Fiscal Deficit [1+2(a)+(b)] -(7+8)(***)","cat1"] = "Fiscal Deficit"

#11 Primary Surplus/Deficit [1+2(a)+2(b)]-[4(a+c+d+e)+5+8]	
df.loc[df["Description"] == "11 Primary Surplus/Deficit [1+2(a)+2(b)]-[4(a+c+d+e)+5+8]","cat1"] = "Primary Surplus/Deficit"

# Replace '--' with '-' then convert
#df["Actuals up to Feb 2026"] = df["Actuals up to Feb 2026"].str.replace("--", "-").astype(float)
# Convert to string, replace the noise, then convert to float
df["Actuals up to Feb 2026"] = df["Actuals up to Feb 2026"].astype(str).str.replace("--", "-").astype(float)
#df["Budget Estimates 2025-26"] = df["Budget Estimates 2025-26"].str.replace("--", "-").astype(float)
# Convert to string -> Replace -> Convert to float
df["Budget Estimates 2025-26"] = (
    df["Budget Estimates 2025-26"]
    .astype(str)
    .str.replace("--", "-")
    .astype(float)
)


df=df.reset_index(drop=True)


df.head(50)

,Description,Budget Estimates 2025-26,Actuals up to Feb 2026,Percentage: Current 2025-26,Percentage: Corresponding 2024-25,State,month,cat2,cat1
0,i)Goods and Service Tax,59704.59,48036.72,80.46,79.26,Telangana,February 2026,Tax Revenue,REVENUE RECEIPTS
1,ii)Stamps and Regstration,19087.26,13775.51,72.17,38.58,Telangana,February 2026,Tax Revenue,REVENUE RECEIPTS
2,iii)Land Revenue,11.20,0.54,4.84,7.39,Telangana,February 2026,Tax Revenue,REVENUE RECEIPTS
3,iv)Sales Tax,37463.90,30628.68,81.76,87.41,Telangana,February 2026,Tax Revenue,REVENUE RECEIPTS
4,v)State Excise Duties,27623.36,20454.14,74.05,65.61,Telangana,February 2026,Tax Revenue,REVENUE RECEIPTS
5,vi)States Share of Union Taxes (*),21195.18,18788.79,88.65,93.79,Telangana,February 2026,Tax Revenue,REVENUE RECEIPTS
6,vii)Other Taxes and duties ($),10233.87,7416.43,72.47,72.10,Telangana,February 2026,Tax Revenue,REVENUE RECEIPTS
7,b)Non-Tax Revenue,31618.77,9105.39,28.80,17.25,Telangana,February 2026,Non-Tax Revenue,REVENUE RECEIPTS
8,c)Grant in aid and Contributions,22782.50,6140.65,26.95,27.78,Telangana,February 2026,Grant in aid and Contributions,REVENUE RECEIPTS
9,a)Recovery of Loans and Advances,1106.93,42.53,3.84,1.11,Telangana,February 2026,Recovery of Loans and Advances,Capital Receipts


In [20]:
df.to_csv("../data/final/data.csv")